In [ ]:
from kubernetes import client, config, watch
import time
import uuid
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timezone

config.load_kube_config()
batch_v1 = client.BatchV1Api()
core_v1 = client.CoreV1Api()

nodes = core_v1.list_node().items
print(f"Cluster has {len(nodes)} nodes.")
for node in nodes:
    print(f"{node.metadata.name}")

In [ ]:
SCHEDULER = "kueue"

def create_job():
    with open("../jobs/testjob.yaml", "r") as f:
        job = yaml.safe_load(f)
    
    job["metadata"]["name"] = f"bench-{uuid.uuid4().hex[:8]}"
    
    if SCHEDULER == "kueue":
        job["metadata"].setdefault("labels", {})
        job["metadata"]["labels"]["kueue.x-k8s.io/queue-name"] = "local-queue"
    else:
        job["metadata"].setdefault("annotations", {})
        job["metadata"]["annotations"]["yunikorn.apache.org/queue"] = "root.default"
    
    return job["metadata"]["name"], job

In [ ]:
NUM_JOBS = 50
submitted_jobs = []

print(f"Submitting {NUM_JOBS} jobs to {SCHEDULER}...")
start_time = time.time()

for i in range(NUM_JOBS):
    job_name, job = create_job()
    batch_v1.create_namespaced_job(namespace="default", body=job)
    submitted_jobs.append(job_name)
    print(f"  Submitted {job_name} ({i+1}/{NUM_JOBS})")

end_time = time.time()
print(f"\nAll jobs submitted in {end_time - start_time:.2f} seconds")